# Abdomen CT — RF-DETR & D-FINE Detection (ultralytics dışı aileler)
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

Bu notebook, `ultralytics` paketinin **kapsamadığı** iki dedektör ailesini eğitir:

| Aile | Paket/Repo | Veri formatı |
|------|-----------|---------------|
| **RF-DETR** (Roboflow) | `pip install rfdetr` | COCO JSON (`train/`, `valid/` klasörleri) |
| **D-FINE** (Peterande) | `git clone` + `requirements.txt` | COCO JSON + YAML config |

`Faz3b_YOLO_Colab_Kaggle.ipynb` (YOLO26/11/10/9/8 + RT-DETR) ile **aynı F1@IoU
değerlendirmesini** (`src.evaluation`) kullanır, böylece sonuçlar karşılaştırılabilir.

> ⚠️ **Önemli — test durumu:** RF-DETR bölümü, Roboflow'un belgelenmiş
> `model.train(dataset_dir=...)` / `model.predict(...)` API'sine dayanır ve
> görece düşük risklidir. **D-FINE bölümü** ise repo'nun YAML config sistemini
> (include zinciri, `train.py -c ...`) bu ortamda **çalıştırılarak doğrulanamadı**
> (GPU/internet erişimi olmayan bir geliştirme ortamında hazırlandı). İlk
> çalıştırmada hata alırsanız D-FINE repo'sundaki `configs/dfine/*.yml` ve
> `configs/dataset/coco_detection.yml` dosyalarını gerçek içerikleriyle
> karşılaştırıp `cell_dfine_config` hücresini buna göre düzeltin.

**Önkoşul:** `Faz3a_VeriHazirlik_YOLO.ipynb` çalıştırılmış olmalı — bu notebook
onun ürettiği `det_data/foldN` (YOLO bbox formatı) çıktısını COCO'ya çevirir.

---

| Adım | Süre (tahmini) |
|------|------|
| Kurulum + COCO dönüşümü | ~10-20 dk |
| RF-DETR eğitim (50 epoch, T4) | ~2-4 saat |
| D-FINE eğitim (~80 epoch, T4) | ~4-8 saat |
| Inference + F1 | ~15 dk |

---
## 0. Ortam Tespiti

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f'Ortam : {env_name}')

---
## 1. Ortam Kurulumu (Colab için)

In [ ]:
if IS_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Kaggle kimlik bilgileri Colab Secrets\'tan yüklendi')
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print('kaggle.json dosyasını seçin:')
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print('kaggle.json yüklendi')

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive bağlandı')
else:
    print('Kaggle/Local ortamı — Colab kurulumu atlandı')

---
## 2. Model Ailesi Seçimi

`MODEL_FAMILY = 'rfdetr'` → Roboflow RF-DETR (Base/Large).
`MODEL_FAMILY = 'dfine'`  → D-FINE (HGNetv2 backbone, n/s/m/l/x boyut).

In [ ]:
MODEL_FAMILY = 'rfdetr'   # 'rfdetr' | 'dfine'

# --- RF-DETR alt seçimi ---
RFDETR_VARIANT = 'base'   # 'base' | 'large'

# --- D-FINE alt seçimi ---
DFINE_SIZE = 'm'           # 'n' | 's' | 'm' | 'l' | 'x'
DFINE_SIZES = ['n', 's', 'm', 'l', 'x']

if MODEL_FAMILY not in ('rfdetr', 'dfine'):
    raise ValueError(f"MODEL_FAMILY 'rfdetr' veya 'dfine' olmalı, alınan: {MODEL_FAMILY}")
if MODEL_FAMILY == 'dfine' and DFINE_SIZE not in DFINE_SIZES:
    raise ValueError(f"DFINE_SIZE seçenekleri: {DFINE_SIZES}")

print(f'Seçilen aile : {MODEL_FAMILY}' +
      (f' ({RFDETR_VARIANT})' if MODEL_FAMILY == 'rfdetr' else f' (hgnetv2_{DFINE_SIZE})'))

---
## 3. Konfigürasyon

In [ ]:
import os, sys, json, shutil, time, subprocess
from pathlib import Path
from typing import List

KAGGLE_DATASET_SLUG = 'abdomen-acikveri'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen-acikveri'
GITHUB_URL          = 'https://github.com/ramazan2020/abdomen1.git'
FOLD = 0

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR   = Path('/kaggle/input') / KAGGLE_DATASET_SLUG
    WORK_DIR   = Path('/kaggle/working')
    DRIVE_BASE = None
elif IS_COLAB:
    DATA_DIR   = Path('/content/kaggle_data')
    WORK_DIR   = Path('/content')
    DRIVE_BASE = Path('/content/drive/MyDrive/Abdomen')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE', r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',  str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',  str(_proje / 'outputs')))
    DRIVE_BASE = None

DET_DATA_DIR  = Path(os.environ.get('ABDOMEN_DET_DATA_DIR', str(WORK_DIR / 'det_data'))) if IS_LOCAL else DATA_DIR / 'det_data'
YOLO_FOLD_DIR = DET_DATA_DIR / f'fold{FOLD}'
COCO_DATA_DIR = WORK_DIR / 'coco_data' / f'fold{FOLD}'
OUTPUT_DIR    = WORK_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ortam         : {env_name}')
print(f'YOLO_FOLD_DIR : {YOLO_FOLD_DIR}  (var={YOLO_FOLD_DIR.exists()})')
print(f'COCO_DATA_DIR : {COCO_DATA_DIR}')

In [ ]:
if IS_LOCAL:
    REPO_DIR = Path('.').resolve()
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.evaluation import f1_at_iou, top5_f1_mean
from src.coco_export import export_coco_split

print('src.evaluation / src.coco_export  ✓')

print(f'Repo : {REPO_DIR}')
print(f'sys.path[0] = {sys.path[0]}')

---
## 4. Veri Kontrolü

`det_data/foldN` **Faz3a_VeriHazirlik_YOLO.ipynb** tarafından üretilir (YOLO bbox
formatı). Bu notebook DICOM'a geri dönmez, sadece bu çıktıyı COCO'ya çevirir.

In [ ]:
if not YOLO_FOLD_DIR.exists():
    raise FileNotFoundError(
        f'YOLO det_data bulunamadı: {YOLO_FOLD_DIR}\n'
        'Önce Faz3a_VeriHazirlik_YOLO.ipynb dosyasını çalıştırın.'
    )

_n_train = len(list((YOLO_FOLD_DIR / 'images' / 'train').glob('*.png')))
_n_val   = len(list((YOLO_FOLD_DIR / 'images' / 'val').glob('*.png')))
if _n_train == 0:
    raise FileNotFoundError(f'PNG bulunamadı: {YOLO_FOLD_DIR}/images/train')

print(f'YOLO det_data mevcut ✓  ({_n_train:,} train · {_n_val:,} val PNG)')

---
## 5. COCO Dönüşümü

`src.coco_export.export_coco_split` mevcut PNG + YOLO `.txt` etiketlerini okuyup
`_annotations.coco.json` üretir — DICOM'a dokunmaz, hızlıdır.

In [ ]:
COCO_TRAIN_DIR = COCO_DATA_DIR / 'train'
COCO_VALID_DIR = COCO_DATA_DIR / 'valid'   # RF-DETR 'valid' adını bekler

if (COCO_TRAIN_DIR / '_annotations.coco.json').exists():
    print(f'COCO verisi mevcut ✓  {COCO_DATA_DIR}')
else:
    print('COCO verisi üretiliyor...')
    t0 = time.time()
    export_coco_split(YOLO_FOLD_DIR, 'train', COCO_TRAIN_DIR, SUPER_CLASSES)
    export_coco_split(YOLO_FOLD_DIR, 'val',   COCO_VALID_DIR, SUPER_CLASSES)
    print(f'COCO verisi hazır ✓  ({time.time()-t0:.0f}s)')

import json as _json
_tr_ann = _json.loads((COCO_TRAIN_DIR / '_annotations.coco.json').read_text())
_vl_ann = _json.loads((COCO_VALID_DIR / '_annotations.coco.json').read_text())
print(f'train: {len(_tr_ann["images"])} görüntü, {len(_tr_ann["annotations"])} annotasyon')
print(f'valid: {len(_vl_ann["images"])} görüntü, {len(_vl_ann["annotations"])} annotasyon')

---
## 6. Kurulum (Seçilen Aileye Göre)

In [ ]:
if MODEL_FAMILY == 'rfdetr':
    print('rfdetr kuruluyor...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rfdetr'], check=True)
    import rfdetr
    print(f'rfdetr ✓  ({getattr(rfdetr, "__version__", "?")})')

elif MODEL_FAMILY == 'dfine':
    DFINE_REPO = WORK_DIR / 'D-FINE'
    if not (DFINE_REPO / '.git').exists():
        print('D-FINE klonlanıyor...')
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/Peterande/D-FINE.git', str(DFINE_REPO)],
                       check=True)
    else:
        print('D-FINE repo mevcut.')

    # requirements.txt'ten torch/torchvision/torchaudio satırlarını çıkar —
    # Kaggle/Colab'ın CUDA'ya uygun önceden kurulu torch'unu BOZMAMAK için.
    _req = DFINE_REPO / 'requirements.txt'
    if _req.exists():
        _filtered = [l for l in _req.read_text().splitlines()
                    if not l.strip().lower().startswith(('torch', 'torchvision', 'torchaudio'))]
        _req_filtered = WORK_DIR / 'dfine_requirements_filtered.txt'
        _req_filtered.write_text('\n'.join(_filtered))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(_req_filtered)], check=True)
        print('D-FINE bağımlılıkları kuruldu (torch hariç) ✓')
    else:
        print(f'[uyarı] requirements.txt bulunamadı: {_req}')

print(f'\nKurulum tamam — aile: {MODEL_FAMILY}')

---
## 7a. RF-DETR Eğitim

Bu hücre yalnızca `MODEL_FAMILY == 'rfdetr'` ise çalışır.

In [ ]:
RFDETR_BEST = None

if MODEL_FAMILY == 'rfdetr':
    from rfdetr import RFDETRBase, RFDETRLarge

    RF_EPOCHS      = 50
    RF_BATCH       = 4
    RF_GRAD_ACCUM  = 4
    RF_LR          = 1e-4
    RF_OUTPUT_DIR  = WORK_DIR / 'runs' / 'rfdetr' / f'fold{FOLD}'

    ModelCls = RFDETRLarge if RFDETR_VARIANT == 'large' else RFDETRBase
    rf_model = ModelCls()

    print(f'RF-DETR eğitimi başlıyor ({RFDETR_VARIANT})...')
    rf_model.train(
        dataset_dir=str(COCO_DATA_DIR),
        epochs=RF_EPOCHS,
        batch_size=RF_BATCH,
        grad_accum_steps=RF_GRAD_ACCUM,
        lr=RF_LR,
        output_dir=str(RF_OUTPUT_DIR),
    )

    # RF-DETR EMA best checkpoint'i kaydeder; isim sürüme göre değişebilir —
    # mevcut .pth dosyaları arasından en yenisini seç.
    _ckpts = sorted(RF_OUTPUT_DIR.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    _best_named = [p for p in _ckpts if 'best' in p.name.lower() and 'ema' in p.name.lower()]
    RFDETR_BEST = (_best_named[-1] if _best_named else (_ckpts[-1] if _ckpts else None))
    print(f'\nRF-DETR checkpoint: {RFDETR_BEST}')

    if IS_COLAB and DRIVE_BASE and RFDETR_BEST:
        _yd = DRIVE_BASE / 'rfdetr_weights'
        _yd.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(RFDETR_BEST), str(_yd / f'fold{FOLD}_{RFDETR_BEST.name}'))
        print(f"Drive'a yedeklendi: {_yd}")
else:
    print('MODEL_FAMILY != "rfdetr" — bu hücre atlandı')

---
## 7b. D-FINE Config Üretimi

⚠️ Bu hücre, D-FINE repo'sunun **mevcut** `configs/dataset/coco_detection.yml` ve
`configs/dfine/dfine_hgnetv2_{size}_coco.yml` dosyalarını **metin olarak**
okuyup, dataset referansını bizim COCO verimize çeviren bir kopya üretir.
Repo güncellenirse (include zinciri/anahtar adları değişirse) bu hücre
güncellenmelidir — çalıştırmadan önce çıktısını (`_dfine_cfg_path`) gözle
kontrol edin.

In [ ]:
_dfine_cfg_path = None

if MODEL_FAMILY == 'dfine':
    DFINE_REPO = WORK_DIR / 'D-FINE'
    _base_dataset_yml = DFINE_REPO / 'configs' / 'dataset' / 'coco_detection.yml'
    _base_model_yml   = DFINE_REPO / 'configs' / 'dfine' / f'dfine_hgnetv2_{DFINE_SIZE}_coco.yml'

    if not _base_dataset_yml.exists() or not _base_model_yml.exists():
        raise FileNotFoundError(
            f'D-FINE config bulunamadı.\n'
            f'  beklenen dataset yaml : {_base_dataset_yml}\n'
            f'  beklenen model yaml   : {_base_model_yml}\n'
            'Repo yapısı değişmiş olabilir — configs/ klasörünü kontrol edin.'
        )

    # ── Özel dataset yaml: img_folder / ann_file / num_classes override ────
    _custom_dataset_yml = DFINE_REPO / 'configs' / 'dataset' / 'abdomen_detection.yml'
    _dataset_src = _base_dataset_yml.read_text()
    import re as _re
    _dataset_src = _re.sub(r'num_classes:\s*\d+', f'num_classes: {len(SUPER_CLASSES)}', _dataset_src)
    _dataset_src = _re.sub(r'remap_mscoco_category:\s*True', 'remap_mscoco_category: False', _dataset_src)
    _dataset_src = _re.sub(r"img_folder:\s*.*", '', _dataset_src)   # eski yolları temizle
    _dataset_src = _re.sub(r"ann_file:\s*.*", '', _dataset_src)
    _custom_dataset_yml.write_text(
        _dataset_src +
        f"\ntrain_dataloader:\n"
        f"  dataset:\n"
        f"    img_folder: {COCO_TRAIN_DIR}\n"
        f"    ann_file: {COCO_TRAIN_DIR / '_annotations.coco.json'}\n"
        f"val_dataloader:\n"
        f"  dataset:\n"
        f"    img_folder: {COCO_VALID_DIR}\n"
        f"    ann_file: {COCO_VALID_DIR / '_annotations.coco.json'}\n"
    )

    # ── Özel model yaml: dataset include'unu bizimkiyle değiştir + output_dir ─
    _model_src = _base_model_yml.read_text()
    _model_src = _model_src.replace('../dataset/coco_detection.yml', '../dataset/abdomen_detection.yml')
    DFINE_OUTPUT_DIR = WORK_DIR / 'runs' / 'dfine' / f'fold{FOLD}'
    _model_src = _re.sub(r'output_dir:\s*.*', f'output_dir: {DFINE_OUTPUT_DIR}', _model_src)
    if 'output_dir:' not in _model_src:
        _model_src += f'\noutput_dir: {DFINE_OUTPUT_DIR}\n'

    _custom_model_yml = DFINE_REPO / 'configs' / 'dfine' / f'dfine_hgnetv2_{DFINE_SIZE}_abdomen.yml'
    _custom_model_yml.write_text(_model_src)

    _dfine_cfg_path = _custom_model_yml
    print(f'Özel dataset yaml : {_custom_dataset_yml}')
    print(f'Özel model yaml   : {_custom_model_yml}')
    print(f'output_dir        : {DFINE_OUTPUT_DIR}')
    print('\n--- model yaml içeriği (ilk 30 satır) ---')
    print('\n'.join(_model_src.splitlines()[:30]))
else:
    print('MODEL_FAMILY != "dfine" — bu hücre atlandı')

---
## 7c. D-FINE Eğitim

In [ ]:
DFINE_BEST = None

if MODEL_FAMILY == 'dfine':
    assert _dfine_cfg_path is not None and _dfine_cfg_path.exists(), \
        'Önce cell_dfine_config çalıştırılmalı.'

    DFINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'D-FINE eğitimi başlıyor — config: {_dfine_cfg_path.name}')

    _train_py = DFINE_REPO / 'train.py'
    if not _train_py.exists():
        raise FileNotFoundError(f'train.py bulunamadı: {_train_py}')

    r = subprocess.run(
        [sys.executable, str(_train_py), '-c', str(_dfine_cfg_path), '--use-amp', '--seed=0'],
        cwd=str(DFINE_REPO),
    )
    if r.returncode != 0:
        raise RuntimeError(f'D-FINE eğitimi başarısız (returncode={r.returncode}) — yukarıdaki çıkışı kontrol edin.')

    _ckpts = sorted(DFINE_OUTPUT_DIR.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    DFINE_BEST = _ckpts[-1] if _ckpts else None
    print(f'\nD-FINE checkpoint: {DFINE_BEST}')

    if IS_COLAB and DRIVE_BASE and DFINE_BEST:
        _yd = DRIVE_BASE / 'dfine_weights'
        _yd.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(DFINE_BEST), str(_yd / f'fold{FOLD}_{DFINE_BEST.name}'))
        print(f"Drive'a yedeklendi: {_yd}")
else:
    print('MODEL_FAMILY != "dfine" — bu hücre atlandı')

---
## 8. Tahmin (Inference) — Validasyon Seti

RF-DETR ve D-FINE için ortak bir `pred_df` üretir (case, image_id, class, score,
x1,y1,x2,y2) — `Faz3b_YOLO_Colab_Kaggle.ipynb` ile aynı şema, karşılaştırılabilir.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

def _parse_stem(stem: str):
    parts = stem.split('_')
    if len(parts) >= 3:
        return int(parts[1]), int(parts[2])
    return int(parts[0]), int(parts[1])

CONF_TH = 0.05
_val_imgs = sorted(COCO_VALID_DIR.glob('*.png'))
print(f'Val inference: {len(_val_imgs)} PNG  (aile={MODEL_FAMILY})')

preds = []

if MODEL_FAMILY == 'rfdetr':
    from rfdetr import RFDETRBase, RFDETRLarge
    assert RFDETR_BEST is not None and RFDETR_BEST.exists(), f'Checkpoint bulunamadı: {RFDETR_BEST}'
    ModelCls = RFDETRLarge if RFDETR_VARIANT == 'large' else RFDETRBase
    rf_infer_model = ModelCls(pretrain_weights=str(RFDETR_BEST))

    for ip in tqdm(_val_imgs, desc='RF-DETR inference'):
        case, image_id = _parse_stem(ip.stem)
        img = Image.open(ip).convert('RGB')
        det = rf_infer_model.predict(img, threshold=CONF_TH)
        for box, sc, cl in zip(det.xyxy, det.confidence, det.class_id):
            preds.append({
                'case': case, 'image_id': image_id, 'class': int(cl),
                'score': float(sc),
                'x1': float(box[0]), 'y1': float(box[1]),
                'x2': float(box[2]), 'y2': float(box[3]),
            })

elif MODEL_FAMILY == 'dfine':
    import torch
    import torchvision.transforms as T

    assert DFINE_BEST is not None and DFINE_BEST.exists(), f'Checkpoint bulunamadı: {DFINE_BEST}'
    if str(DFINE_REPO) not in sys.path:
        sys.path.insert(0, str(DFINE_REPO))
    from src.core import YAMLConfig   # D-FINE repo içi import (tools/inference/torch_inf.py paterni)

    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    cfg = YAMLConfig(str(_dfine_cfg_path), resume=str(DFINE_BEST))
    _ckpt = torch.load(str(DFINE_BEST), map_location='cpu')
    _state = _ckpt.get('ema', {}).get('module') or _ckpt['model']
    cfg.model.load_state_dict(_state)

    class _DFineWrapper(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.model = cfg.model.deploy()
            self.postprocessor = cfg.postprocessor.deploy()
        def forward(self, images, orig_sizes):
            return self.postprocessor(self.model(images), orig_sizes)

    dfine_model = _DFineWrapper().to(_device).eval()
    _tf = T.Compose([T.Resize((640, 640)), T.ToTensor()])

    for ip in tqdm(_val_imgs, desc='D-FINE inference'):
        case, image_id = _parse_stem(ip.stem)
        img = Image.open(ip).convert('RGB')
        w0, h0 = img.size
        x = _tf(img)[None].to(_device)
        sizes = torch.tensor([[w0, h0]]).to(_device)
        with torch.no_grad():
            labels, boxes, scores = dfine_model(x, sizes)
        labels, boxes, scores = labels[0], boxes[0], scores[0]
        for box, sc, cl in zip(boxes.cpu().numpy(), scores.cpu().numpy(), labels.cpu().numpy()):
            if sc < CONF_TH:
                continue
            preds.append({
                'case': case, 'image_id': image_id, 'class': int(cl),
                'score': float(sc),
                'x1': float(box[0]), 'y1': float(box[1]),
                'x2': float(box[2]), 'y2': float(box[3]),
            })

pred_df = pd.DataFrame(preds)
print(f'Tahmin: {len(pred_df):,} kutu  |  {pred_df["case"].nunique() if not pred_df.empty else 0} vaka')

---
## 9. Ground Truth Yükleme (YOLO val label'larından)

In [ ]:
gt_rows = []
_val_lbl_dir = YOLO_FOLD_DIR / 'labels' / 'val'
for lp in sorted(_val_lbl_dir.glob('*.txt')):
    ip = COCO_VALID_DIR / (lp.stem + '.png')
    if not ip.exists():
        continue
    W, H = Image.open(ip).size
    case, image_id = _parse_stem(lp.stem)
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cl = int(p[0]); cx, cy, w, h = map(float, p[1:5])
        gt_rows.append({
            'case': case, 'image_id': image_id, 'class': cl,
            'x1': (cx - w/2)*W, 'y1': (cy - h/2)*H,
            'x2': (cx + w/2)*W, 'y2': (cy + h/2)*H,
        })

gt_df = pd.DataFrame(gt_rows)
print(f'GT: {len(gt_df):,} bbox  |  {gt_df["case"].nunique() if not gt_df.empty else 0} vaka')

---
## 10. Değerlendirme — F1 Skoru (Faz3b ile karşılaştırılabilir)

In [ ]:
if pred_df.empty:
    print('Prediction boş — inference adımını kontrol edin.')
else:
    top5   = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)

    print(f'=== {MODEL_FAMILY.upper()} — Fold {FOLD} Sonuçları ===')
    print(f'Top-5 Mean F1      : {top5["top5_mean_f1"]:.4f}')
    print(f'Macro F1 @ IoU=0.3 : {detail["macro_f1"]:.4f}')
    print(f'Micro F1 @ IoU=0.3 : {detail["micro_f1"]:.4f}')
    print()
    for cls_id, cls_name in enumerate(SUPER_CLASSES):
        if cls_id in detail['per_class']:
            m = detail['per_class'][cls_id]
            print(f'  {cls_id} {cls_name:<30}  '
                  f'P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}  '
                  f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')

---
## 11. Sonuçları Kaydet

In [ ]:
if pred_df.empty:
    print('Prediction boş, kaydetme atlandı.')
else:
    _tag = MODEL_FAMILY if MODEL_FAMILY == 'rfdetr' else f'dfine_{DFINE_SIZE}'
    pred_df.to_csv(OUTPUT_DIR / f'{_tag}_fold{FOLD}_pred.csv', index=False)

    summary = {
        'fold'          : FOLD,
        'family'        : MODEL_FAMILY,
        'top5_mean_f1'  : top5['top5_mean_f1'],
        'macro_f1_03'   : detail['macro_f1'],
        'micro_f1_03'   : detail['micro_f1'],
        'per_class_03'  : {
            SUPER_CLASSES[cid]: {
                k: round(v, 4) if isinstance(v, float) else v
                for k, v in m.items()
            }
            for cid, m in detail['per_class'].items()
        },
    }
    (OUTPUT_DIR / f'{_tag}_fold{FOLD}_summary.json').write_text(
        json.dumps(summary, indent=2, ensure_ascii=False)
    )
    print(f'Çıktılar: {OUTPUT_DIR}')
    for f in sorted(OUTPUT_DIR.glob(f'{_tag}_*')):
        print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

    if IS_COLAB and DRIVE_BASE:
        _dst = DRIVE_BASE / 'output'
        _dst.mkdir(parents=True, exist_ok=True)
        for f in OUTPUT_DIR.glob(f'{_tag}_*'):
            shutil.copy2(str(f), str(_dst / f.name))
        print(f"Drive'a kopyalandı: {_dst}")